# 🔩 Data Augmentation for Steel Defect Classification (NEU): Performance Comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtzai/course_ai_for_industrial_control_students/blob/main/notebooks/03_steel_defects_classification/n14_steel_defect_neu_multiclass_augmentation.ipynb)

> **En Colab:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU** (el notebook también funciona en CPU, solo tarda más).

In this notebook we train **two identical CNNs** on the six NEU defect classes, one **without** and one **with data augmentation**, and compare them in a **fair** way:

- ✅ **Same training images** for both models (same indices, only the transform changes).
- ✅ **Fixed validation set** (never augmented, never used for training) to follow the training.
- ✅ **Separate test set** used only once at the end.
- ✅ **Same seed** → same split and same initial weights.
- ✅ The split is done by **blocks of consecutive images** (NEU images come from the same strips; neighbours can be near-duplicates).

We repeat the comparison with **two training-set sizes** (few images per class and all of them) because the effect of augmentation usually depends on how much data you have.

📚 Theory: data-efficient methods and data augmentation (Parte 3, slides 97–104); overfitting and how to reduce it (Parte 2, slides 50–53 and 134–135).

## 📋 Important: Execution Order
Run the cells **in order from top to bottom**. The training cell trains 4 models (2 sizes × with/without augmentation): about 1–2 min on a Colab GPU and around 5 min on a laptop CPU.

In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

import importlib.util
import subprocess
import sys

missing = [package for package in ('kagglehub', 'gdown') if importlib.util.find_spec(package) is None]
if missing:
    if importlib.util.find_spec('pip') is None:
        subprocess.run([sys.executable, '-m', 'ensurepip'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('Running in Colab' if IN_COLAB else 'Running locally')


In [ ]:
# 📦 Import Required Libraries
import os
import time
import copy
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from sklearn.metrics import confusion_matrix, accuracy_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

QUICK_RUN = os.environ.get('QUICK_RUN', '0') == '1'   # only for automatic tests
IMG_SIZE = 128
BATCH_SIZE = 32
NUM_WORKERS = 2 if IN_COLAB else 0   # parallel data loading in Colab; 0 is safest on Windows/Jupyter
LEARNING_RATE = 1e-3
EPOCHS = 3 if QUICK_RUN else 30              # minimum number of epochs
MIN_STEPS = 30 if QUICK_RUN else 600         # small training sets get more epochs so that every model gets >= MIN_STEPS updates
TRAIN_SIZES = [30, 'all']                    # images per class used for training ('all' = 180)
print('QUICK_RUN =', QUICK_RUN, '| EPOCHS =', EPOCHS, '| TRAIN_SIZES =', TRAIN_SIZES)

In [ ]:
# 📥 Download the NEU Surface Defect Database into ./data
# Sources tried in order (no Kaggle login needed for the first two):
#   1) kagglehub: 'kaustubhdikshit/neu-surface-defect-database' (public Kaggle mirror of NEU-DET)
#   2) Official Google Drive link of NEU-DET published by the dataset authors (via gdown)
#   3) Manual: download NEU-DET (or NEU-CLS) and unzip it anywhere inside ./data
import os
import zipfile
from pathlib import Path
from PIL import Image

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)
os.environ.setdefault('KAGGLEHUB_CACHE', str((DATA_DIR / 'kagglehub').resolve()))  # keep everything in ./data

NEU_CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
NEU_KAGGLE_HANDLE = 'kaustubhdikshit/neu-surface-defect-database'
NEU_GDRIVE_ID = '1qrdZlaDi272eA79b0uCwwqPrm2Q_WI3k'   # NEU-DET.zip, official link from the authors' page


def list_neu_images(root):
    """All NEU images below `root`. Class and index come from the file name: <class>_<index>.jpg"""
    files = {}
    for p in Path(root).rglob('*.jpg'):
        cls, _, idx = p.stem.rpartition('_')
        if cls in NEU_CLASSES and idx.isdigit():
            files.setdefault(p.name, p)  # de-duplicate by file name
    return sorted(files.values())


def download_neu():
    if len(list_neu_images(DATA_DIR)) >= 1800:
        return DATA_DIR
    errors = []
    try:
        import kagglehub
        path = kagglehub.dataset_download(NEU_KAGGLE_HANDLE)
        if len(list_neu_images(path)) >= 1800:
            return Path(path)
        errors.append(f'kagglehub: fewer than 1800 NEU images found in {path}')
    except Exception as e:  # noqa: BLE001
        errors.append(f'kagglehub: {e}')
    try:
        import gdown
        zip_path = DATA_DIR / 'NEU-DET.zip'
        if not zip_path.exists():
            gdown.download(id=NEU_GDRIVE_ID, output=str(zip_path), quiet=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)
        if len(list_neu_images(DATA_DIR)) >= 1800:
            return DATA_DIR
        errors.append('Google Drive: fewer than 1800 NEU images found after extraction')
    except Exception as e:  # noqa: BLE001
        errors.append(f'Google Drive: {e}')
    raise RuntimeError(
        'Could not download NEU automatically.\n' + '\n'.join(errors) +
        '\nManual option: download NEU-DET from https://www.kaggle.com/datasets/' + NEU_KAGGLE_HANDLE +
        ' or from http://faculty.neu.edu.cn/songkc/en/zdylm/263270/list/index.htm and unzip it inside ./data')


NEU_ROOT = download_neu()
print(f'NEU images found: {len(list_neu_images(NEU_ROOT))} in {NEU_ROOT}')

In [ ]:
# 🧱 Load the images (grayscale, resized) and build a reproducible train / validation / test split
def load_neu(root, img_size):
    paths = list_neu_images(root)
    X = np.stack([np.array(Image.open(p).convert('L').resize((img_size, img_size), Image.BILINEAR))
                  for p in paths])                                       # (N, H, W) uint8
    y = np.array([NEU_CLASSES.index(p.stem.rpartition('_')[0]) for p in paths])
    idx = np.array([int(p.stem.rpartition('_')[2]) for p in paths])     # image number inside its class
    return X, y, idx


def blocked_split(y, idx, block_size=10, fractions=(0.6, 0.2, 0.2), seed=42):
    """Stratified split by blocks of consecutive image numbers.
    NEU images are crops of the same steel strips and neighbouring numbers can be very similar,
    so we keep each block of `block_size` consecutive images in ONE split (avoids optimistic test results)."""
    rng = np.random.default_rng(seed)
    parts = {'train': [], 'val': [], 'test': []}
    for c in np.unique(y):
        members = np.where(y == c)[0]
        blocks = (idx[members] - 1) // block_size
        order = rng.permutation(np.unique(blocks))
        n_tr = int(round(fractions[0] * len(order)))
        n_va = int(round(fractions[1] * len(order)))
        for name, sel in [('train', order[:n_tr]), ('val', order[n_tr:n_tr + n_va]), ('test', order[n_tr + n_va:])]:
            parts[name].extend(members[np.isin(blocks, sel)])
    return tuple(np.array(sorted(parts[k])) for k in ('train', 'val', 'test'))


X_all, y_all, idx_all = load_neu(NEU_ROOT, IMG_SIZE)
train_idx, val_idx, test_idx = blocked_split(y_all, idx_all, seed=SEED)
print('All images:', X_all.shape, '| classes:', NEU_CLASSES)
for name, ids in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    print(f'{name:>5}: {len(ids):4d} images | per class: {np.bincount(y_all[ids], minlength=6).tolist()}')

## 🔄 Two preprocessing pipelines

The **augmented** pipeline only uses transformations that make sense for a steel strip seen by a fixed camera:

| Transformation | Why it is plausible on the line |
|---|---|
| Horizontal and vertical flip | a defect looks the same mirrored; the strip can run in either direction |
| Rotation ±15° | small misalignment of the strip / camera |
| Random resized crop (70–100 % area) | defect position and size inside the field of view change |
| Translation ±10 % | defect not centred |
| Brightness / contrast ±20 % | illumination and material changes (a known difficulty of NEU) |

We do **not** use colour jitter (images are gray), 180° rotations or strong distortions that could change what the defect looks like.
Validation and test images always use the **basic** pipeline.

In [ ]:
AUGMENTATION = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(degrees=15, fill=0.5),
    v2.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1), antialias=True),
    v2.RandomAffine(degrees=0, translate=(0.1, 0.1), fill=0.5),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
])

X_float = torch.from_numpy(X_all.astype(np.float32) / 255.0).unsqueeze(1)   # (N, 1, H, W) in [0, 1]
MEAN, STD = X_float[train_idx].mean().item(), X_float[train_idx].std().item()


class NEUDataset(Dataset):
    """Returns (normalised image, label). If `augment` is True the random transforms are applied on the fly,
    so every epoch the network sees a different version of each training image."""
    def __init__(self, ids, augment=False):
        self.ids, self.augment = np.asarray(ids), augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, k):
        i = self.ids[k]
        x = X_float[i]
        if self.augment:
            x = AUGMENTATION(x)
        return (x - MEAN) / STD, int(y_all[i])


val_loader = DataLoader(NEUDataset(val_idx), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(NEUDataset(test_idx), batch_size=BATCH_SIZE, shuffle=False)
print(f'Normalisation (train statistics): mean={MEAN:.3f} std={STD:.3f}')

In [ ]:
# 👀 Visualize Data Augmentation Effects
torch.manual_seed(SEED)
fig, axes = plt.subplots(3, 7, figsize=(12, 5.5))
for r, c in enumerate([NEU_CLASSES.index('scratches'), NEU_CLASSES.index('inclusion'), NEU_CLASSES.index('patches')]):
    i = train_idx[y_all[train_idx] == c][0]
    axes[r, 0].imshow(X_float[i, 0], cmap='gray', vmin=0, vmax=1)
    axes[r, 0].set_title('Original' if r == 0 else '', fontsize=9)
    axes[r, 0].set_ylabel(NEU_CLASSES[c])
    for k in range(1, 7):
        axes[r, k].imshow(AUGMENTATION(X_float[i])[0], cmap='gray', vmin=0, vmax=1)
        if r == 0:
            axes[r, k].set_title(f'Augmented {k}', fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Left: original training image. Right: 6 random augmented versions')
plt.tight_layout()
plt.show()

## 🧠 Model and training function
Same CNN as in n13. For each experiment we create the model **after** `torch.manual_seed(SEED)`, so the models with and without augmentation start from **exactly the same weights**.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, kernel_size=3, padding=1), nn.BatchNorm2d(c_out),
                         nn.ReLU(), nn.MaxPool2d(2))


class SteelDefectCNN(nn.Module):
    """4 x (Conv-BN-ReLU-MaxPool) + Global Average Pooling + Dropout + Linear."""
    def __init__(self, num_outputs=6, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 128))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(128, num_outputs))

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))

In [ ]:
def subset_per_class(ids, n_per_class, seed=SEED):
    """First the split, then the subset: the small training set is a subset of the same training images."""
    if n_per_class == 'all':
        return np.asarray(ids)
    rng = np.random.default_rng(seed)
    return np.sort(np.concatenate([rng.choice(ids[y_all[ids] == c], n_per_class, replace=False)
                                   for c in range(len(NEU_CLASSES))]))


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, preds, labels = 0.0, [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += criterion(logits, yb).item() * len(yb)
            preds.append(logits.argmax(1).cpu()); labels.append(yb.cpu())
    preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    return total_loss / len(labels), accuracy_score(labels, preds), preds, labels


def train_model(train_ids, augment, name):
    torch.manual_seed(SEED)                      # same initial weights and same batch order for both models
    loader = DataLoader(NEUDataset(train_ids, augment), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    model = SteelDefectCNN(num_outputs=len(NEU_CLASSES)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    epochs = max(EPOCHS, math.ceil(MIN_STEPS / len(loader)))
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val, best_state = np.inf, None
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total, correct, n = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(yb); correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
        va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion)
        for k, v in zip(hist, [total / n, correct / n, va_loss, va_acc]):
            hist[k].append(v)
        if va_loss < best_val:
            best_val, best_state, best_epoch = va_loss, copy.deepcopy(model.state_dict()), epoch + 1
    hist['time'] = time.time() - start
    hist['best_epoch'] = best_epoch
    model.load_state_dict(best_state)
    print(f'{name:<32} epochs={epochs:3d} time={hist["time"]:6.1f}s best epoch={best_epoch:3d} '
          f'final train acc={hist["train_acc"][-1]:.3f} val acc={hist["val_acc"][-1]:.3f}')
    return model, hist

## 🏋️ Train Both Models: With and Without Augmentation (for each training-set size)

In [ ]:
results, histories, models = [], {}, {}
criterion = nn.CrossEntropyLoss()
for size in TRAIN_SIZES:
    ids = subset_per_class(train_idx, size)
    for augment in [False, True]:
        name = f'{size}/class - ' + ('WITH aug' if augment else 'WITHOUT aug')
        model, hist = train_model(ids, augment, name)
        te_loss, te_acc, te_pred, te_true = evaluate(model, test_loader, criterion)
        k = (size, augment)
        histories[k], models[k] = hist, model
        results.append({'images/class': size, 'augmentation': augment, 'train images': len(ids),
                        'epochs': len(hist['train_acc']), 'best epoch': hist['best_epoch'],
                        'train acc (last)': hist['train_acc'][-1], 'best val acc': max(hist['val_acc']),
                        'train-val gap (last)': hist['train_acc'][-1] - hist['val_acc'][-1],
                        'test acc': te_acc, 'time (s)': hist['time'], 'pred': te_pred, 'true': te_true})

table = pd.DataFrame(results).drop(columns=['pred', 'true'])
print(table.round(3).to_string(index=False))

## 📈 Training History Visualization and Analysis
Look at: (1) the **gap** between train and validation accuracy, (2) how **noisy** the validation curve is, (3) the **loss** curves.
Note that the training accuracy *with* augmentation is measured on augmented images, which are harder than the originals.

In [ ]:
fig, axes = plt.subplots(len(TRAIN_SIZES), 2, figsize=(12, 4 * len(TRAIN_SIZES)), squeeze=False)
for r, size in enumerate(TRAIN_SIZES):
    for augment, color in [(False, 'tab:blue'), (True, 'tab:red')]:
        h = histories[(size, augment)]
        lab = 'WITH aug' if augment else 'WITHOUT aug'
        axes[r, 0].plot(h['train_acc'], color=color, ls='--', label=f'{lab} - train')
        axes[r, 0].plot(h['val_acc'], color=color, label=f'{lab} - validation')
        axes[r, 1].plot(h['train_loss'], color=color, ls='--', label=f'{lab} - train')
        axes[r, 1].plot(h['val_loss'], color=color, label=f'{lab} - validation')
    axes[r, 0].set_title(f'Accuracy - {size} images/class'); axes[r, 1].set_title(f'Loss - {size} images/class')
    axes[r, 1].set_yscale('log')
    for ax in axes[r]:
        ax.set_xlabel('Epoch'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 🏆 Final Test Set Evaluation

In [ ]:
fig, axes = plt.subplots(len(TRAIN_SIZES), 2, figsize=(13, 5.5 * len(TRAIN_SIZES)), squeeze=False)
for r, size in enumerate(TRAIN_SIZES):
    for c, augment in enumerate([False, True]):
        res = next(x for x in results if x['images/class'] == size and x['augmentation'] == augment)
        cm = confusion_matrix(res['true'], res['pred'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Reds' if augment else 'Blues', cbar=False,
                    xticklabels=NEU_CLASSES, yticklabels=NEU_CLASSES, ax=axes[r, c])
        axes[r, c].set_title(f'{size}/class - {"WITH" if augment else "WITHOUT"} augmentation\n'
                             f'test accuracy = {res["test acc"]:.3f}')
        axes[r, c].set_xlabel('Predicted'); axes[r, c].set_ylabel('Actual')
        axes[r, c].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

for size in TRAIN_SIZES:
    a = next(x for x in results if x['images/class'] == size and not x['augmentation'])
    b = next(x for x in results if x['images/class'] == size and x['augmentation'])
    print(f'{size}/class: test acc {a["test acc"]:.3f} -> {b["test acc"]:.3f} ({100 * (b["test acc"] - a["test acc"]):+.1f} pp) | '
          f'time {a["time (s)"]:.0f}s -> {b["time (s)"]:.0f}s ({100 * (b["time (s)"] / a["time (s)"] - 1):+.0f} %)')

## 📝 Summary: Data Augmentation for Industrial Vision Systems

**What to look at in your own results** (they depend on the run, the data and the transformations; an improvement is *not* guaranteed):
- Test accuracy with and without augmentation for each training-set size.
- The train–validation gap and the shape of the validation curve.
- The time overhead of generating augmented images on the fly.
- Which classes improve or get worse in the confusion matrix.

**Why augmentation can help:** each epoch the network sees a slightly different version of every image, so it is harder to memorise the training set and it can learn **invariances** (position, orientation, illumination) that also exist on the real line.

**Why it can hurt:** if a transformation produces images that never happen in production, or destroys the feature that defines a defect, the model learns from wrong examples.

### 🔧 Things you can try (and report in the quiz)
- Change `TRAIN_SIZES` (e.g. `[10, 30, 'all']`).
- Remove one transformation at a time from `AUGMENTATION` and see which one matters.
- Add a transformation that does **not** make sense for steel and observe the effect.

➡️ Next notebook (**n17**): instead of training from scratch, start from a network **pre-trained on ImageNet (fine-tuning)**.